In [2]:
!pip install -q openpyxl tqdm rapidfuzz

from openpyxl import load_workbook
from tqdm.auto import tqdm
from rapidfuzz import fuzz
import re
import html
import unicodedata

# =========================
# 1. File paths
# =========================

EXPORT_FILE = "/content/franckenachlass_4.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"
CORRECTED_GEO_FILE = "/content/GeoNames_updated.xlsx"
OUTPUT_FILE = "/content/franckenachlass_5.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx"

HEADER_ROW = 1

TARGET_REPLACE_COLUMNS = [
    "creator_linked_new",
    "receiver_linked_new",
    "sender_place_linked_new",
    "receiver_place_linked_new"
]

SENDER_PLACE_COL = "sender_place"
RECEIVER_PLACE_COL = "receiver_place"

SENDER_PLACE_LINKED_NEW = "sender_place_linked_new"
RECEIVER_PLACE_LINKED_NEW = "receiver_place_linked_new"

FALLBACK_ORIGINAL_LINKED_COLS = {
    "creator_linked_new": "creator_linked",
    "receiver_linked_new": "receiver_linked",
    "sender_place_linked_new": "sender_place_linked",
    "receiver_place_linked_new": "receiver_place_linked",
}

OLD_GEO_COL_CANDIDATES = [
    "LODgeoName",
    "LODgeoName_inDB",
    "LODgeoName_original"
]

NEW_GEO_URI_COL_CANDIDATES = [
    "LODgeoName_Updated",
    "LODgeoName_updated",
    "LODgeoName_uri",
    "LODgeoName_URI"
]

NEW_GEO_LABEL_COL_CANDIDATES = [
    "geoLabel_final2_final",
    "geoLabel_final",
    "geoLabel_final2",
    "geoLabel",
    "GeoNames_label"
]

FUZZY_PLACE_THRESHOLD = 90
ONE_WORD_PLACE_THRESHOLD = 97
AMBIGUITY_SCORE_GAP = 2


# =========================
# 2. Regex
# =========================

GEONAMES_ANCHOR_RE = re.compile(
    r'(?P<start><a\b[^>]*\bhref=["\']'
    r'(?P<href>https?://(?:www\.|sws\.)?geonames\.org/(?P<geoid>\d+)(?:/[^"\']*)?)'
    r'["\'][^>]*>)'
    r'(?P<label>.*?)'
    r'(?P<end></a>)',
    re.IGNORECASE | re.DOTALL
)


# =========================
# 3. Helper functions
# =========================

def get_header_positions(ws):
    positions = {}
    for col in range(1, ws.max_column + 1):
        value = ws.cell(HEADER_ROW, col).value
        if value is not None:
            header = str(value).strip()
            positions.setdefault(header, []).append(col)
    return positions


def get_first_col(header_positions, col_name):
    if col_name not in header_positions:
        raise ValueError(f"Missing required column: {col_name}")
    return header_positions[col_name][0]


def get_first_available_col(header_positions, candidates, required=True):
    for col_name in candidates:
        if col_name in header_positions:
            return col_name, header_positions[col_name][0]

    if required:
        raise ValueError(f"Missing all candidate columns: {candidates}")

    return None, None


def get_cell_value_or_hyperlink(cell):
    if cell.hyperlink and cell.hyperlink.target:
        return cell.hyperlink.target
    return cell.value


def is_blank(value):
    return value is None or str(value).strip() == ""


def normalize_space(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def extract_geonames_id(value):
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(r"\d+", text):
        return text

    m = re.search(r"geonames\.org/(\d+)", text, flags=re.IGNORECASE)

    if m:
        return m.group(1)

    return None


def normalize_geonames_uri(uri):
    geoid = extract_geonames_id(uri)

    if geoid:
        return f"https://sws.geonames.org/{geoid}"

    if uri is None:
        return ""

    return str(uri).strip().rstrip("/")


def normalize_old_geonames_key(uri):
    geoid = extract_geonames_id(uri)

    if geoid:
        return geoid

    return str(uri).strip().rstrip("/") if uri is not None else None


def build_geonames_anchor(uri, label):
    uri = html.escape(normalize_geonames_uri(uri), quote=True)
    label = html.escape(str(label).strip(), quote=False)
    return f'<p><a href="{uri}">{label}</a> [GeoNames]</p>'


def append_geonames_anchor_if_missing(cell_value, anchor_html, new_geoid):
    if is_blank(cell_value):
        return anchor_html, True

    cell_text = str(cell_value)

    existing_ids = set()

    for m in GEONAMES_ANCHOR_RE.finditer(cell_text):
        existing_id = extract_geonames_id(m.group("href"))
        if existing_id:
            existing_ids.add(existing_id)

    if new_geoid in existing_ids:
        return cell_text, False

    return cell_text + anchor_html, True


def get_or_create_target_col(ws, header_positions, new_col_name):
    if new_col_name in header_positions:
        return header_positions[new_col_name][0]

    fallback_col_name = FALLBACK_ORIGINAL_LINKED_COLS.get(new_col_name)

    if fallback_col_name is None:
        raise ValueError(f"No fallback column defined for {new_col_name}")

    if fallback_col_name not in header_positions:
        raise ValueError(
            f"Missing both '{new_col_name}' and fallback '{fallback_col_name}'"
        )

    fallback_idx = header_positions[fallback_col_name][0]

    new_idx = ws.max_column + 1
    ws.cell(HEADER_ROW, new_idx).value = new_col_name

    for row in range(HEADER_ROW + 1, ws.max_row + 1):
        ws.cell(row, new_idx).value = ws.cell(row, fallback_idx).value

    header_positions[new_col_name] = [new_idx]

    print(f"Created '{new_col_name}' from '{fallback_col_name}'")

    return new_idx


def base_place_name(label):
    if label is None:
        return ""

    text = str(label).strip()
    text = re.sub(r"\s*\([^)]*\)\s*$", "", text).strip()

    return text


# =========================
# NEW AMBIGUITY RESOLVER CODE
# =========================

def before_first_bracket(text):
    """
    For corrected GeoNames label:
    'Glaucha (Halle (Saale), Kreisfreie Stadt Halle, ...)' -> 'Glaucha'
    """
    if text is None:
        return ""

    text = str(text).strip()
    text = text.split("(", 1)[0].strip()
    return text


def before_first_comma(text):
    """
    For main place/geo name:
    'Glaucha, Halle (Saale), Kreisfreie Stadt Halle, ...' -> 'Glaucha'
    """
    if text is None:
        return ""

    text = str(text).strip()
    text = text.split(",", 1)[0].strip()
    return text


def normalized_prefix_match(main_place, corrected_label):
    """
    Extra condition for ambiguous cases:
    If label before first bracket in corrected geoLabel_final2
    matches label before first comma in main geo/place name,
    then use that record.
    """
    main_prefix = normalize_place_for_match(before_first_comma(main_place))
    corrected_prefix = normalize_place_for_match(before_first_bracket(corrected_label))

    if not main_prefix or not corrected_prefix:
        return False

    return main_prefix == corrected_prefix


# =========================
# End new ambiguity resolver code
# =========================


def normalize_place_for_match(text):
    if text is None:
        return ""

    text = str(text).strip()
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    text = text.strip("[]")

    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    text = text.lower()

    text = re.sub(r"\bschloss\b", "castle", text)
    text = re.sub(r"\bburg\b", "castle", text)

    text = re.sub(
        r"\b(city of|town of|village of|municipality of|county of|district of)\b",
        " ",
        text
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)

    text = re.sub(
        r"\b(city|town|village|municipality|county|district|province|state|region|country|place)\b",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def place_match_score(main_place, corrected_label):
    main_norm = normalize_place_for_match(main_place)

    corrected_full = normalize_place_for_match(corrected_label)
    corrected_base = normalize_place_for_match(base_place_name(corrected_label))

    if not main_norm or not corrected_base:
        return 0

    if main_norm == corrected_base or main_norm == corrected_full:
        return 100

    main_words = main_norm.split()
    base_words = corrected_base.split()

    if len(main_words) >= 2 and len(base_words) >= 2:
        if corrected_base in main_norm or main_norm in corrected_base:
            return 96

    score_base = fuzz.token_set_ratio(main_norm, corrected_base)
    score_full = fuzz.token_set_ratio(main_norm, corrected_full)
    score_sort_base = fuzz.token_sort_ratio(main_norm, corrected_base)
    score_sort_full = fuzz.token_sort_ratio(main_norm, corrected_full)

    return max(score_base, score_full, score_sort_base, score_sort_full)


def is_safe_score(main_place, corrected_label, score):
    main_norm = normalize_place_for_match(main_place)
    corrected_base = normalize_place_for_match(base_place_name(corrected_label))

    main_words = main_norm.split()
    corrected_words = corrected_base.split()

    if len(main_words) <= 1 or len(corrected_words) <= 1:
        return score >= ONE_WORD_PLACE_THRESHOLD

    return score >= FUZZY_PLACE_THRESHOLD


def add_to_multimap(mapping, key, record):
    if not key:
        return
    mapping.setdefault(key, []).append(record)

# =========================
# 4. Load corrected GeoNames mapping
# =========================

geo_wb = load_workbook(CORRECTED_GEO_FILE, data_only=True)

geo_ws = None
geo_headers = None

for ws in geo_wb.worksheets:
    headers = get_header_positions(ws)

    has_new_uri = any(c in headers for c in NEW_GEO_URI_COL_CANDIDATES)
    has_new_label = any(c in headers for c in NEW_GEO_LABEL_COL_CANDIDATES)

    if has_new_uri and has_new_label:
        geo_ws = ws
        geo_headers = headers
        break

if geo_ws is None:
    raise ValueError("Could not find corrected GeoNames sheet with required columns.")

print("Using corrected GeoNames sheet:", geo_ws.title)

old_geo_col_used, old_geo_idx = get_first_available_col(
    geo_headers,
    OLD_GEO_COL_CANDIDATES,
    required=False
)

new_geo_uri_col_used, new_geo_uri_idx = get_first_available_col(
    geo_headers,
    NEW_GEO_URI_COL_CANDIDATES
)

new_geo_label_col_used, new_geo_label_idx = get_first_available_col(
    geo_headers,
    NEW_GEO_LABEL_COL_CANDIDATES
)

print("Using old GeoNames column:", old_geo_col_used)
print("Using new GeoNames URI column:", new_geo_uri_col_used)
print("Using new GeoNames label column:", new_geo_label_col_used)

geo_map_by_id = {}
geo_map_by_old_url = {}
geo_records = []
duplicate_old_ids = []

insert_by_place_key = {}

for row in range(HEADER_ROW + 1, geo_ws.max_row + 1):
    old_geo = None

    if old_geo_idx is not None:
        old_cell = geo_ws.cell(row, old_geo_idx)
        old_geo = get_cell_value_or_hyperlink(old_cell)

    new_uri_cell = geo_ws.cell(row, new_geo_uri_idx)
    new_label_cell = geo_ws.cell(row, new_geo_label_idx)

    new_geo_uri = get_cell_value_or_hyperlink(new_uri_cell)
    new_geo_label = new_label_cell.value

    if is_blank(new_geo_uri) or is_blank(new_geo_label):
        continue

    old_id = extract_geonames_id(old_geo)
    old_key = normalize_old_geonames_key(old_geo)

    new_id = extract_geonames_id(new_geo_uri)
    new_uri_norm = normalize_geonames_uri(new_geo_uri)
    new_label_clean = str(new_geo_label).strip()
    new_base_clean = base_place_name(new_label_clean)

    if not new_id:
        continue

    record = {
        "corrected_geo_row": row,
        "old_geo": str(old_geo).strip() if old_geo is not None else "",
        "old_geo_id": old_id,
        "new_geo_id": new_id,
        "new_geo_uri": new_uri_norm,
        "new_geo_label": new_label_clean,
        "new_geo_base_label": new_base_clean,
    }

    geo_records.append(record)

    if old_id:
        if old_id in geo_map_by_id:
            duplicate_old_ids.append(old_id)
        geo_map_by_id[old_id] = record

    if old_key:
        geo_map_by_old_url[old_key] = record

    for key_text in [new_label_clean, new_base_clean]:
        key = normalize_place_for_match(key_text)
        add_to_multimap(insert_by_place_key, key, record)

print("Loaded GeoNames correction records:", len(geo_records))
print("Loaded GeoNames correction records by old ID:", len(geo_map_by_id))
print("Duplicate old GeoNames IDs found:", len(set(duplicate_old_ids)))
print("Exact place insertion keys:", len(insert_by_place_key))


# =========================
# 5. Load export workbook
# =========================

export_wb = load_workbook(EXPORT_FILE)

export_ws = None
export_headers = None

for ws in export_wb.worksheets:
    headers = get_header_positions(ws)

    has_place_cols = (
        SENDER_PLACE_COL in headers
        and RECEIVER_PLACE_COL in headers
    )

    has_linked_cols = (
        SENDER_PLACE_LINKED_NEW in headers
        or FALLBACK_ORIGINAL_LINKED_COLS[SENDER_PLACE_LINKED_NEW] in headers
    )

    if has_place_cols and has_linked_cols:
        export_ws = ws
        export_headers = headers
        break

if export_ws is None:
    raise ValueError(
        "Could not find export sheet with sender_place, receiver_place, and linked columns."
    )

print("Using export sheet:", export_ws.title)

sender_place_idx = get_first_col(export_headers, SENDER_PLACE_COL)
receiver_place_idx = get_first_col(export_headers, RECEIVER_PLACE_COL)

target_col_indexes = []

for col_name in TARGET_REPLACE_COLUMNS:
    col_idx = get_or_create_target_col(
        export_ws,
        export_headers,
        col_name
    )
    target_col_indexes.append((col_name, col_idx))

sender_place_linked_new_idx = get_first_col(export_headers, SENDER_PLACE_LINKED_NEW)
receiver_place_linked_new_idx = get_first_col(export_headers, RECEIVER_PLACE_LINKED_NEW)

print("Using target columns:")
for name, idx in target_col_indexes:
    print(f"  {name}: Excel column {idx}")

print("Sender place column:", sender_place_idx)
print("Receiver place column:", receiver_place_idx)


# =========================
# 6. Replace existing GeoNames anchors
# =========================

replace_log = []
unmatched_geo_log = []
used_record_rows = set()
total_replaced = 0


def replace_geonames_in_cell(cell_value, row_num, col_name):
    if cell_value is None or not isinstance(cell_value, str):
        return cell_value, []

    if "geonames.org" not in cell_value.lower():
        return cell_value, []

    changes = []

    def repl(match):
        old_href = match.group("href")
        old_label = match.group("label")
        old_geoid = extract_geonames_id(old_href)
        old_key = normalize_old_geonames_key(old_href)

        record = None

        if old_geoid:
            record = geo_map_by_id.get(old_geoid)

        if record is None and old_key:
            record = geo_map_by_old_url.get(old_key)

        if record is None:
            unmatched_geo_log.append([
                row_num,
                col_name,
                old_geoid,
                old_href,
                old_label
            ])
            return match.group(0)

            new_href = record["new_geo_uri"]
            new_label = record["new_geo_label"]

            # ---------------------------------------------------------
            # Do NOT replace if the only URI difference is a trailing /
            # and the label is already the same.
            #
            # Examples treated as equivalent:
            # https://sws.geonames.org/12345
            # https://sws.geonames.org/12345/
            # ---------------------------------------------------------
            old_href_compare = html.unescape(str(old_href)).strip().rstrip("/")
            new_href_compare = html.unescape(str(new_href)).strip().rstrip("/")

            old_label_compare = html.unescape(str(old_label)).strip()
            new_label_compare = html.unescape(str(new_label)).strip()

            if (
                old_href_compare == new_href_compare
                and old_label_compare == new_label_compare
            ):
                # No meaningful correction needed.
                # Keep the existing anchor exactly unchanged.
                used_record_rows.add(record["corrected_geo_row"])
                return match.group(0)


            # Otherwise perform the actual replacement
            escaped_href = html.escape(new_href, quote=True)
            escaped_label = html.escape(new_label, quote=False)

            changes.append([
                row_num,
                col_name,
                record["corrected_geo_row"],
                old_geoid,
                old_href,
                old_label,
                new_href,
                new_label
            ])

            used_record_rows.add(record["corrected_geo_row"])

            return f'<a href="{escaped_href}">{escaped_label}</a>'

    new_value = GEONAMES_ANCHOR_RE.sub(repl, cell_value)

    return new_value, changes


for col_name, col_idx in target_col_indexes:
    for row in tqdm(
        range(HEADER_ROW + 1, export_ws.max_row + 1),
        desc=f"Replacing GeoNames in {col_name}"
    ):
        cell = export_ws.cell(row, col_idx)
        old_value = cell.value

        new_value, changes = replace_geonames_in_cell(
            old_value,
            row,
            col_name
        )

        if changes:
            cell.value = new_value
            total_replaced += len(changes)
            replace_log.extend(changes)

print("Total GeoNames anchors replaced:", total_replaced)
print("GeoNames anchors not found in corrected_geo:", len(unmatched_geo_log))


# =========================
# 7. Insert missing GeoNames by sender_place / receiver_place
# =========================

insert_log = []
ambiguous_insert_log = []
no_match_place_log = []
used_insert_record_rows = set()
total_inserted = 0

match_cache = {}


def find_geo_records_for_place(place_value):
    place_text = normalize_space(place_value)

    if is_blank(place_text):
        return {
            "records": [],
            "match_type": "blank",
            "score": 0
        }

    cache_key = normalize_place_for_match(place_text)

    if cache_key in match_cache:
        return match_cache[cache_key]

    exact_records = insert_by_place_key.get(cache_key, [])

    if exact_records:
        unique = {}

        for rec in exact_records:
            unique[rec["new_geo_id"]] = rec

        result = {
            "records": list(unique.values()),
            "match_type": "exact_normalized",
            "score": 100
        }

        match_cache[cache_key] = result
        return result

    scored = []

    for rec in geo_records:
        score = place_match_score(place_text, rec["new_geo_label"])

        if is_safe_score(place_text, rec["new_geo_label"], score):
            scored.append((score, rec))

    if not scored:
        result = {
            "records": [],
            "match_type": "no_safe_match",
            "score": 0
        }

        match_cache[cache_key] = result
        return result

    scored.sort(key=lambda x: x[0], reverse=True)

    best_score = scored[0][0]

    close_matches = [
        (score, rec)
        for score, rec in scored
        if best_score - score <= AMBIGUITY_SCORE_GAP
    ]

    unique = {}

    for score, rec in close_matches:
        unique[rec["new_geo_id"]] = (score, rec)

    records = [item[1] for item in unique.values()]

    # =========================
    # NEW AMBIGUITY RESOLVER CODE
    # =========================
    # If multiple fuzzy matches are close, resolve ambiguity when:
    # corrected geoLabel_final2 before first "("
    # matches main geo/place label before first ","
    #
    # Example:
    # corrected label: Glaucha (Halle (Saale), Kreisfreie Stadt Halle, ...)
    # main place:      Glaucha, Halle (Saale), Kreisfreie Stadt Halle, ...
    # matched prefix:  Glaucha

    if len(records) > 1:
        prefix_matched_records = [
            rec for rec in records
            if normalized_prefix_match(place_text, rec["new_geo_label"])
        ]

        unique_prefix = {}

        for rec in prefix_matched_records:
            unique_prefix[rec["new_geo_id"]] = rec

        prefix_matched_records = list(unique_prefix.values())

        if len(prefix_matched_records) == 1:
            result = {
                "records": prefix_matched_records,
                "match_type": "fuzzy_prefix_resolved",
                "score": best_score
            }

            match_cache[cache_key] = result
            return result

    # =========================
    # End new ambiguity resolver code
    # =========================

    result = {
        "records": records,
        "match_type": "fuzzy",
        "score": best_score
    }

    match_cache[cache_key] = result
    return result


def insert_geo_record_into_cell(row, target_col_idx, target_col_name, records, place_value, match_type, score):
    global total_inserted

    if not records:
        return

    unique = {}

    for rec in records:
        unique[rec["new_geo_id"]] = rec

    records = list(unique.values())

    if len(records) > 1:
        ambiguous_insert_log.append([
            row,
            target_col_name,
            place_value,
            match_type,
            score,
            "; ".join(
                f'{rec["corrected_geo_row"]}: {rec["new_geo_label"]} -> {rec["new_geo_uri"]}'
                for rec in records
            )
        ])
        return

    rec = records[0]

    anchor = build_geonames_anchor(
        rec["new_geo_uri"],
        rec["new_geo_label"]
    )

    cell = export_ws.cell(row, target_col_idx)
    old_cell_value = cell.value

    new_cell_value, did_insert = append_geonames_anchor_if_missing(
        old_cell_value,
        anchor,
        rec["new_geo_id"]
    )

    if did_insert:
        cell.value = new_cell_value
        total_inserted += 1

        used_record_rows.add(rec["corrected_geo_row"])
        used_insert_record_rows.add(rec["corrected_geo_row"])

        insert_log.append([
            row,
            target_col_name,
            place_value,
            match_type,
            score,
            rec["corrected_geo_row"],
            rec["old_geo_id"],
            rec["old_geo"],
            rec["new_geo_id"],
            rec["new_geo_uri"],
            rec["new_geo_label"],
            old_cell_value,
            new_cell_value
        ])


for row in tqdm(
    range(HEADER_ROW + 1, export_ws.max_row + 1),
    desc="Inserting GeoNames by place"
):
    sender_place = export_ws.cell(row, sender_place_idx).value

    if not is_blank(sender_place):
        result = find_geo_records_for_place(sender_place)

        if result["records"]:
            insert_geo_record_into_cell(
                row=row,
                target_col_idx=sender_place_linked_new_idx,
                target_col_name="sender_place_linked_new",
                records=result["records"],
                place_value=sender_place,
                match_type=result["match_type"],
                score=result["score"]
            )
        else:
            no_match_place_log.append([
                row,
                "sender_place",
                sender_place,
                result["match_type"],
                result["score"]
            ])

    receiver_place = export_ws.cell(row, receiver_place_idx).value

    if not is_blank(receiver_place):
        result = find_geo_records_for_place(receiver_place)

        if result["records"]:
            insert_geo_record_into_cell(
                row=row,
                target_col_idx=receiver_place_linked_new_idx,
                target_col_name="receiver_place_linked_new",
                records=result["records"],
                place_value=receiver_place,
                match_type=result["match_type"],
                score=result["score"]
            )
        else:
            no_match_place_log.append([
                row,
                "receiver_place",
                receiver_place,
                result["match_type"],
                result["score"]
            ])

print("Total new GeoNames anchors inserted:", total_inserted)
print("Ambiguous GeoNames insert cases skipped:", len(ambiguous_insert_log))


# =========================
# 8. Add log sheets
# =========================

log_sheet_names = [
    "geo_replace_log",
    "geo_insert_log",
    "geo_ambig_log",
    "geo_not_in_corr",
    "geo_duplicate_ids",
    "geo_no_place_match",
    "geo_not_applied"
]

for sheet_name in log_sheet_names:
    if sheet_name in export_wb.sheetnames:
        del export_wb[sheet_name]


ws_replace = export_wb.create_sheet("geo_replace_log")
ws_replace.append([
    "export_row",
    "column",
    "corrected_geo_row",
    "old_geonames_id",
    "old_geonames_href",
    "old_geonames_label",
    "new_geonames_href",
    "new_geonames_label"
])

for item in replace_log:
    ws_replace.append(item)


ws_insert = export_wb.create_sheet("geo_insert_log")
ws_insert.append([
    "export_row",
    "target_column",
    "place_value",
    "match_type",
    "match_score",
    "corrected_geo_row",
    "old_geonames_id",
    "old_geonames_href",
    "new_geonames_id",
    "new_geonames_href",
    "new_geonames_label",
    "old_cell_value",
    "new_cell_value"
])

for item in insert_log:
    ws_insert.append(item)


ws_ambiguous = export_wb.create_sheet("geo_ambig_log")
ws_ambiguous.append([
    "export_row",
    "target_column",
    "place_value",
    "match_type",
    "match_score",
    "possible_geonames_records"
])

for item in ambiguous_insert_log:
    ws_ambiguous.append(item)


ws_unmatched = export_wb.create_sheet("geo_not_in_corr")
ws_unmatched.append([
    "export_row",
    "column",
    "old_geonames_id",
    "old_geonames_href",
    "old_geonames_label"
])

seen_unmatched = set()

for item in unmatched_geo_log:
    key = tuple(item)
    if key in seen_unmatched:
        continue

    seen_unmatched.add(key)
    ws_unmatched.append(item)


ws_dupes = export_wb.create_sheet("geo_duplicate_ids")
ws_dupes.append(["duplicate_old_geonames_id"])

for geoid in sorted(set(duplicate_old_ids)):
    ws_dupes.append([geoid])


ws_no_match = export_wb.create_sheet("geo_no_place_match")
ws_no_match.append([
    "export_row",
    "place_column",
    "place_value",
    "match_type",
    "match_score"
])

seen_no_match = set()

for item in no_match_place_log:
    key = tuple(str(x) for x in item)

    if key in seen_no_match:
        continue

    seen_no_match.add(key)
    ws_no_match.append(item)


ws_not_applied = export_wb.create_sheet("geo_not_applied")
ws_not_applied.append([
    "corrected_geo_row",
    "old_geonames_id",
    "old_geonames_href",
    "new_geonames_id",
    "new_geonames_href",
    "new_geonames_label",
    "new_geonames_base_label",
    "reason"
])

for rec in geo_records:
    if rec["corrected_geo_row"] not in used_record_rows:
        ws_not_applied.append([
            rec["corrected_geo_row"],
            rec["old_geo_id"],
            rec["old_geo"],
            rec["new_geo_id"],
            rec["new_geo_uri"],
            rec["new_geo_label"],
            rec["new_geo_base_label"],
            "Not replaced and not inserted. No safe sender_place/receiver_place match, skipped as ambiguous, or not present in export."
        ])


for ws_log in [
    ws_replace,
    ws_insert,
    ws_ambiguous,
    ws_unmatched,
    ws_dupes,
    ws_no_match,
    ws_not_applied
]:
    for col in range(1, ws_log.max_column + 1):
        ws_log.column_dimensions[ws_log.cell(1, col).column_letter].width = 34


# =========================
# 9. Save safely
# =========================

for ws in export_wb.worksheets:
    ws.sheet_state = "visible"

export_wb.active = export_wb.sheetnames.index(export_ws.title)

if export_wb.views:
    export_wb.views[0].activeTab = export_wb.sheetnames.index(export_ws.title)
    export_wb.views[0].firstSheet = 0

export_wb.save(OUTPUT_FILE)

print("Done.")
print("Saved file:", OUTPUT_FILE)

Using corrected GeoNames sheet: Sheet1
Using old GeoNames column: LODgeoName_inDB
Using new GeoNames URI column: LODgeoName_Updated
Using new GeoNames label column: geoLabel_final
Loaded GeoNames correction records: 466
Loaded GeoNames correction records by old ID: 458
Duplicate old GeoNames IDs found: 0
Exact place insertion keys: 902
Using export sheet: 40427936
Using target columns:
  creator_linked_new: Excel column 39
  receiver_linked_new: Excel column 40
  sender_place_linked_new: Excel column 41
  receiver_place_linked_new: Excel column 42
Sender place column: 4
Receiver place column: 6


Replacing GeoNames in creator_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing GeoNames in receiver_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing GeoNames in sender_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Replacing GeoNames in receiver_place_linked_new:   0%|          | 0/324 [00:00<?, ?it/s]

Total GeoNames anchors replaced: 0
GeoNames anchors not found in corrected_geo: 0


Inserting GeoNames by place:   0%|          | 0/324 [00:00<?, ?it/s]

Total new GeoNames anchors inserted: 8
Ambiguous GeoNames insert cases skipped: 28
Done.
Saved file: /content/franckenachlass_5.xls_Wed_Sep_09_07_12_33_2026part_1.xlsx
